# Sealed expert evaluation

Evaluate one already-frozen checkpoint against the three expert-agreement splits. This notebook never trains a model or selects a checkpoint.

In [ ]:
import os
from pathlib import Path

from ai4mars import acquire_frozen_checkpoint, run_sealed_expert_evaluation

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

configured_dataset = os.environ.get('AI4MARS_DATASET_ROOT')
if configured_dataset:
    DATASET_ROOT = Path(configured_dataset)
else:
    local_dataset = PROJECT_ROOT / 'data' / 'raw' / 'ai4mars' / 'ai4mars-dataset-merged-0.6'
    matches = [local_dataset] if local_dataset.is_dir() else list(Path('/kaggle/input').glob('**/ai4mars-dataset-merged-0.6'))
    if len(matches) != 1:
        raise RuntimeError('Set AI4MARS_DATASET_ROOT to the extracted AI4Mars merged 0.6 directory.')
    DATASET_ROOT = matches[0]

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'reproduction' / 'paper_deeplabv3plus_kaggle_p100.yaml'
MANIFEST_ROOT = PROJECT_ROOT / 'artifacts' / 'manifests'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'expert-evaluation'
CHECKPOINT_PATH = PROJECT_ROOT / 'artifacts' / 'ai4mars-paper-reproduction' / 'frozen' / 'deeplabv3plus-tesla-p100-seed42-best-val-miou.pth'
acquire_frozen_checkpoint(CHECKPOINT_PATH)

In [ ]:
run_dir = run_sealed_expert_evaluation(
    config_path=CONFIG_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    dataset_root=DATASET_ROOT,
    manifest_root=MANIFEST_ROOT,
    output_root=OUTPUT_ROOT,
    device='auto',
)
print(f'Completed sealed evaluation: {run_dir}')

In [ ]:
import json

report = json.loads((run_dir / 'artifacts' / 'expert_evaluation.json').read_text(encoding='utf-8'))
for split_name, metrics in report['splits'].items():
    print(f"{split_name}: pixel accuracy={metrics['pixel_accuracy']:.4f}, mIoU={metrics['mean_iou']:.4f}")
    for item in metrics['per_class']:
        print(f"  class {item['class_index']}: IoU={item['iou']} recall={item['recall']}")
print('Confusion-matrix CSV and PNG artifacts are in', run_dir / 'artifacts')